# Día 10 · Unidad 11 — Pipelines, clasificación y riesgos del ML aplicado

**Objetivos de aprendizaje**
- Construir un `ColumnTransformer` que combine imputación, escalado y codificación *one-hot* según el tipo de columna.
- Encadenar preprocesado y modelo en un `Pipeline` de scikit-learn, y entender por qué eso evita fugas de datos.
- Entrenar y comparar dos clasificadores (`LogisticRegression`, `RandomForestClassifier`) para predecir la anulación de una póliza de auto.
- Interpretar accuracy, precision, recall y matriz de confusión en términos de negocio (falsos positivos/negativos).
- Ajustar hiperparámetros con `RandomizedSearchCV`.
- Reconocer y evitar los riesgos principales del ML aplicado: sobreajuste, *data leakage*, variables mal definidas y sesgos.
- Usar un asistente de IA para explicar un pipeline y para revisar código en busca de *leakage* — sin darlo por bueno sin comprobarlo.

**Duración estimada:** 120-150 minutos.

**Requisitos previos:** `01_teoria_regresion_lineal.ipynb` de este mismo día, pandas (Día 7-8).

## 0. El problema y el dataset

Vamos a construir un modelo que prediga si una póliza de auto se va a **anular** (`anulada`), a partir de las características de la póliza y del asegurado. Es el mismo problema — y el mismo dataset — del proyecto final opcional del curso (`proyecto_final/`), así que este notebook os deja el terreno preparado si decidís hacerlo.

Usamos `data/raw/polizas_auto_anulacion.csv` en vez de un dataset genérico de terceros: es sintético, del propio dominio del curso, y está documentado en `data/generators/README.md` (incluye nulos en `potencia_cv` y un outlier en `prima_anual`, **a propósito**, para practicar justo lo que vamos a hacer aquí).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

polizas = pd.read_csv("../data/raw/polizas_auto_anulacion.csv")
print(polizas.shape)
polizas.head(3)

In [ ]:
polizas.isna().sum()[lambda s: s > 0]

In [ ]:
polizas["prima_anual"].describe()

Como se documenta en `data/generators/README.md`, `potencia_cv` tiene nulos a propósito, y `prima_anual` tiene un outlier a propósito (el máximo, 45.000€, está muy lejos del percentil 75 de ~530€). Ambas cosas hay que tratarlas **dentro** del flujo de preprocesado, no a mano y de forma ad-hoc — es justo lo que resuelve un `Pipeline`.

También comprobamos el balance de la variable objetivo, porque condiciona qué métricas tienen sentido más adelante:

In [ ]:
polizas["anulada"].value_counts(normalize=True).rename("proporcion")

Aproximadamente un 11-12% de las pólizas se anulan: es un problema con **clases desbalanceadas**. Con un dataset así, un modelo que siempre prediga "no se anula" ya acertaría ~88% de las veces — por eso el `accuracy` a secas puede ser engañoso (lo retomamos en la sección de métricas).

> 💡 El material heredado de este notebook (edición anterior del curso) resolvía el desbalanceo con `SMOTE` (de la librería `imblearn`). Este curso mantiene el alcance de ML en scikit-learn puro, sin librerías adicionales — así que en vez de SMOTE usamos el parámetro `class_weight="balanced"`, disponible de forma nativa en `LogisticRegression` y `RandomForestClassifier`: pondera cada clase de forma inversamente proporcional a su frecuencia, sin generar filas sintéticas nuevas.

## 1. `ColumnTransformer`: preprocesado distinto según el tipo de columna

Nuestras columnas se dividen en dos grupos con necesidades distintas:

- **Numéricas** (`edad_conductor`, `antiguedad_carnet`, `antiguedad_vehiculo`, `potencia_cv`, `km_anuales`, `num_siniestros_3_anios`, `antiguedad_poliza_meses`, `prima_anual`): hay que imputar el nulo de `potencia_cv` y escalar, porque `LogisticRegression` es sensible a la escala de las variables.
- **Categóricas** (`tipo_vehiculo`, `uso`, `provincia`, `canal_venta`, `forma_pago`): no tienen nulos, pero hay que convertirlas a numéricas con *one-hot encoding* para que un modelo lineal o un árbol las puedan usar.

`ColumnTransformer` aplica una transformación distinta a cada subconjunto de columnas y junta el resultado en una única matriz.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

columnas_numericas = [
    "edad_conductor", "antiguedad_carnet", "antiguedad_vehiculo", "potencia_cv",
    "km_anuales", "num_siniestros_3_anios", "antiguedad_poliza_meses", "prima_anual",
]
columnas_categoricas = ["tipo_vehiculo", "uso", "provincia", "canal_venta", "forma_pago"]

# recibo_impagado_alguna_vez ya es booleana (0/1 de facto) - la pasamos tal cual
columna_booleana = ["recibo_impagado_alguna_vez"]

preprocesado_numerico = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="median")),
    # RobustScaler usa mediana/IQR en vez de media/desviacion tipica: es menos
    # sensible que StandardScaler al outlier a proposito de prima_anual.
    ("escalar", RobustScaler()),
])

preprocesado_categorico = Pipeline(steps=[
    ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocesador = ColumnTransformer(transformers=[
    ("numericas", preprocesado_numerico, columnas_numericas),
    ("categoricas", preprocesado_categorico, columnas_categoricas),
    ("booleana", "passthrough", columna_booleana),
])

Un par de detalles que conviene entender línea a línea (y que retomamos en el bloque de IA):

- `SimpleImputer(strategy="median")` rellena `potencia_cv` con la **mediana calculada solo con los datos de entrenamiento** — nunca con la mediana de todo el dataset, porque eso sería una fuga de datos (sección 4).
- `OneHotEncoder(handle_unknown="ignore")` evita que el pipeline falle si en producción aparece una categoría nueva (p. ej. una provincia que no estaba en el entrenamiento): la codifica como todo ceros en vez de lanzar un error.
- `sparse_output=False` es el nombre correcto del parámetro en scikit-learn ≥ 1.2 (antes se llamaba `sparse`, y ya está eliminado en versiones recientes — si veis `sparse=False` en documentación antigua, es la misma idea con el nombre viejo).
- `"passthrough"` dentro de un `ColumnTransformer` significa "deja estas columnas tal cual, sin transformarlas".

## 2. `Pipeline`: encadenar preprocesado y modelo

Un `Pipeline` combina el `preprocesador` con un estimador final en un único objeto con la misma interfaz (`.fit()`, `.predict()`, `.score()`) que cualquier modelo suelto. Probamos dos algoritmos: una `LogisticRegression` (modelo lineal, interpretable) y un `RandomForestClassifier` (conjunto de árboles, captura relaciones no lineales sin necesidad de escalar).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pipeline_logistica = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipeline_bosque = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", RandomForestClassifier(class_weight="balanced", random_state=42)),
])

## 3. `train_test_split` y entrenamiento

Separamos entrenamiento y prueba **antes** de tocar los datos con cualquier transformación — el `Pipeline` se encarga de que el preprocesado se ajuste (`.fit()`) solo con el train y se aplique (`.transform()`) igual en ambos. Usamos `stratify=y` para que la proporción de pólizas anuladas sea la misma en train y test, dado el desbalanceo.

In [ ]:
from sklearn.model_selection import train_test_split

X = polizas.drop(columns=["poliza_id", "anulada"])
y = polizas["anulada"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipeline_logistica.fit(X_train, y_train)
pipeline_bosque.fit(X_train, y_train)
print("Ambos pipelines entrenados.")

## 4. Métricas de clasificación, en términos de negocio

Con clases desbalanceadas, el `accuracy` (proporción de aciertos) no basta: un modelo que prediga siempre "no se anula" tendría ~88% de accuracy sin ser útil. Miramos también **precision** y **recall** de la clase `anulada`, y la matriz de confusión completa.

En este problema, cada celda de la matriz de confusión tiene un significado de negocio muy concreto:

| | Predicho: no se anula | Predicho: se anula |
|---|---|---|
| **Real: no se anula** | acierto | **falso positivo** — contactamos a un cliente que no iba a anular (coste de una gestión comercial de más) |
| **Real: se anula** | **falso negativo** — no actuamos sobre un cliente que se va (coste de perder la póliza) |
| acierto |

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

def evaluar(pipeline, nombre):
    y_pred = pipeline.predict(X_test)
    print(f"--- {nombre} ---")
    print(f"accuracy:  {accuracy_score(y_test, y_pred):.3f}")
    print(f"precision: {precision_score(y_test, y_pred):.3f}  (de las que predije 'se anula', cuantas acerte)")
    print(f"recall:    {recall_score(y_test, y_pred):.3f}  (de las que realmente se anulan, cuantas detecte)")
    print(f"f1:        {f1_score(y_test, y_pred):.3f}")
    return y_pred

y_pred_logistica = evaluar(pipeline_logistica, "Regresion logistica")
y_pred_bosque = evaluar(pipeline_bosque, "Random forest")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, y_pred, nombre in zip(axes, [y_pred_logistica, y_pred_bosque], ["Regresion logistica", "Random forest"]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["no anulada", "anulada"]).plot(ax=ax, colorbar=False)
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

La elección entre priorizar **precision** o **recall** es una decisión de negocio, no técnica: si contactar a un cliente para retenerlo es barato, conviene un `recall` alto aunque baje la `precision` (mejor contactar de más que perder pólizas). Si la gestión de retención es cara o intrusiva, se prioriza `precision`. Con `class_weight="balanced"` empujamos ambos modelos hacia un `recall` más alto de la clase minoritaria de lo que darían "de fábrica" — a costa de más falsos positivos.

**Para ti:** vuelve a entrenar `pipeline_logistica` sin `class_weight="balanced"` (quítalo del constructor) y compara el `recall` de la clase `anulada`. ¿Cambia mucho? ¿Qué pasaría si la aseguradora decide que prefiere un `recall` aún más alto — qué le pedirías cambiar al modelo?

## 5. Ajuste de hiperparámetros con `RandomizedSearchCV`

En vez de probar combinaciones de hiperparámetros a mano, `RandomizedSearchCV` prueba un número fijo de combinaciones aleatorias dentro de un espacio de búsqueda, evaluando cada una con validación cruzada (`cv`). Lo aplicamos al `RandomForestClassifier`, ajustando el `Pipeline` completo — así el preprocesado también se re-ajusta correctamente en cada partición de la validación cruzada, sin fugas.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

espacio_busqueda = {
    "modelo__n_estimators": [100, 200, 300],
    "modelo__max_depth": [4, 8, 12, None],
    "modelo__min_samples_leaf": [1, 5, 10],
}

busqueda = RandomizedSearchCV(
    pipeline_bosque,
    param_distributions=espacio_busqueda,
    n_iter=8,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=1,
)
busqueda.fit(X_train, y_train)

print("Mejores hiperparametros:", busqueda.best_params_)
print(f"Mejor f1 en validacion cruzada: {busqueda.best_score_:.3f}")

mejor_pipeline = busqueda.best_estimator_
_ = evaluar(mejor_pipeline, "Random forest (ajustado)")

Nota el prefijo `modelo__` en las claves de `espacio_busqueda`: es la sintaxis de scikit-learn para referirse al hiperparámetro `n_estimators` del paso llamado `"modelo"` dentro del `Pipeline` — así se pueden buscar hiperparámetros de cualquier paso (incluido el preprocesado) sin ambigüedad.

**Para ti:** añade `"preprocesado__numericas__imputar__strategy": ["median", "mean"]` al `espacio_busqueda` para que la búsqueda también decida cómo imputar `potencia_cv`. ¿Gana la mediana o la media en este dataset?

## 6. Riesgos del machine learning aplicado

Un modelo que "funciona" en el notebook no es lo mismo que un modelo fiable para tomar decisiones reales. Cuatro riesgos que hay que revisar siempre, con criterio profesional — la IA puede ayudar a detectarlos, pero no sustituye la revisión.

### 6.1 Sobreajuste (*overfitting*)

Ya lo vimos con regresión polinómica: un modelo demasiado flexible memoriza el ruido del entrenamiento en vez de aprender la relación real. En clasificación se detecta igual: comparando la métrica en train y en test.

In [ ]:
from sklearn.metrics import f1_score as f1

for nombre, pipeline in [("Regresion logistica", pipeline_logistica), ("Random forest", pipeline_bosque), ("Random forest (ajustado)", mejor_pipeline)]:
    f1_train = f1(y_train, pipeline.predict(X_train))
    f1_test = f1(y_test, pipeline.predict(X_test))
    print(f"{nombre:28s}  f1 train = {f1_train:.3f}  |  f1 test = {f1_test:.3f}  |  brecha = {f1_train - f1_test:.3f}")

Un `RandomForestClassifier` sin limitar `max_depth` casi siempre sobreajusta bastante (los árboles pueden crecer hasta separar cada observación de entrenamiento). Fíjate en la brecha train/test del bosque sin ajustar frente al ajustado por `RandomizedSearchCV`, que sí limita `max_depth` y `min_samples_leaf`: una brecha grande es la señal de que hay que simplificar el modelo (menos profundidad, más regularización, más datos), no de que "el modelo es bueno porque acierta mucho en entrenamiento".

### 6.2 Fuga de datos (*data leakage*)

Es el riesgo más traicionero porque el código **no da ningún error** — el modelo simplemente parece mejor de lo que realmente es, porque durante el entrenamiento ha visto, directa o indirectamente, información que en producción no tendría disponible.

El ejemplo clásico: escalar (o imputar, o seleccionar variables) con **todo** el dataset antes de dividir en train/test, en vez de ajustar el transformador solo con el train.

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- MAL: fuga de datos ---
# El StandardScaler se ajusta (fit) con TODO el dataset, incluido lo que
# despues sera el conjunto de test. La media y la desviacion tipica que
# usa para escalar ya "conocen" la distribucion del test.
X_numerico = polizas[columnas_numericas].fillna(polizas[columnas_numericas].median())

scaler_con_fuga = StandardScaler()
X_escalado_con_fuga = scaler_con_fuga.fit_transform(X_numerico)  # fit sobre TODO X

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_escalado_con_fuga, y, test_size=0.25, random_state=42, stratify=y
)

modelo_con_fuga = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelo_con_fuga.fit(X_train_leak, y_train_leak)
f1_con_fuga = f1(y_test_leak, modelo_con_fuga.predict(X_test_leak))

# --- BIEN: el scaler solo ve el train ---
X_train_ok, X_test_ok, y_train_ok, y_test_ok = train_test_split(
    X_numerico, y, test_size=0.25, random_state=42, stratify=y
)
scaler_correcto = StandardScaler().fit(X_train_ok)  # fit SOLO sobre train
X_train_ok_esc = scaler_correcto.transform(X_train_ok)
X_test_ok_esc = scaler_correcto.transform(X_test_ok)

modelo_correcto = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelo_correcto.fit(X_train_ok_esc, y_train_ok)
f1_correcto = f1(y_test_ok, modelo_correcto.predict(X_test_ok_esc))

print(f"f1 con fuga (scaler ajustado con todo el dataset): {f1_con_fuga:.4f}")
print(f"f1 correcto (scaler ajustado solo con train):      {f1_correcto:.4f}")

Con un `StandardScaler` y este dataset concreto, la diferencia numérica es pequeña — y eso es precisamente lo peligroso: la fuga **no siempre "se nota"** en la métrica. Con otros pasos del pipeline (selección de variables usando la variable objetivo de todo el dataset, imputación con estadísticos de todo el dataset, o un remuestreo tipo SMOTE aplicado antes del split) el efecto puede ser mucho más grande y inflar la métrica de forma engañosa, haciendo parecer un modelo mucho mejor de lo que será en producción.

La regla que evita el problema de raíz, siempre: **todo paso que "aprenda" algo de los datos (`fit`) se ajusta solo con el conjunto de entrenamiento**, y esa es exactamente la garantía que da usar un `Pipeline` de scikit-learn dentro de `train_test_split`/`cross_val_score`/`RandomizedSearchCV`, en vez de transformar los datos "a mano" antes de dividir.

### 6.3 Variables mal definidas

No toda fuga de datos viene de un `fit()` mal colocado — a veces la propia variable no debería estar disponible en el momento de predecir. Algunos ejemplos para este problema concreto:

- Una variable como "número de días hasta la baja" sería un caso evidente: solo existe **después** de que la póliza se anule, así que no puede usarse como predictor (es casi una copia disfrazada de la variable objetivo).
- `antiguedad_poliza_meses` es legítima **siempre que se calcule a fecha de la predicción**, no a fecha de la anulación — si por error se calculara "antigüedad hasta que se dio de baja", volveríamos a tener una fuga del tipo anterior.
- `recibo_impagado_alguna_vez` es un buen predictor legítimo (impago previo es señal real de riesgo de anulación, según `data/generators/README.md`) — pero solo si su ventana temporal es "hasta la fecha de la predicción", no "durante toda la vida de la póliza incluyendo después del momento en que se anuló".

La forma de detectar esto no es estadística, es conceptual: para cada variable, preguntarse **"¿esta información existiría de verdad en el momento de hacer la predicción, en producción?"**. Ninguna métrica de scikit-learn te avisa de esto — es responsabilidad de quien construye el modelo entender el proceso de negocio.

### 6.4 Sesgos

Un modelo entrenado con datos históricos reproduce los patrones (y los sesgos) que hubiera en esos datos. Por ejemplo, si `provincia` o `canal_venta` correlacionan con variables sensibles de forma indirecta, el modelo puede acabar penalizando a un colectivo concreto sin que ninguna variable "sensible" esté explícitamente en el dataset — es el problema de las *proxy variables*.

Para un colectivo colegiado con responsabilidad profesional, esto no es un detalle técnico: un modelo de anulación que se traduzca en "a quién llamamos para retener" o, peor, en decisiones de tarificación, está sujeto a las mismas exigencias de no discriminación que cualquier otro criterio actuarial. Antes de usar un modelo así en producción hace falta, como mínimo:

- Revisar el rendimiento del modelo **por subgrupos** (p. ej. por provincia), no solo en global.
- Preguntarse si alguna variable es un proxy de algo que no debería influir en la decisión.
- Documentar la decisión y su justificación — igual que se documentaría un supuesto actuarial ante un regulador.

## 7. Con ayuda de la IA: entender pipelines y detectar leakage

En esta unidad, el bloque de IA tiene dos usos típicos: **pedir que explique un pipeline línea a línea** (para verificarlo ejecutando cada paso por separado) y **pedir que revise un fragmento con leakage** (para comprobar si lo detecta o no — y aprender de cualquiera de los dos resultados).

### Prompt sugerido 1 — explicar el pipeline

In [ ]:
prompt_explicar_pipeline = """
Contexto: tengo este Pipeline de scikit-learn en un notebook de clasificacion
(predecir si una poliza de auto se anula):

preprocesador = ColumnTransformer(transformers=[
    ("numericas", Pipeline([("imputar", SimpleImputer(strategy="median")),
                             ("escalar", RobustScaler())]), columnas_numericas),
    ("categoricas", Pipeline([("codificar", OneHotEncoder(handle_unknown="ignore",
                                                            sparse_output=False))]),
                              columnas_categoricas),
    ("booleana", "passthrough", columna_booleana),
])

pipeline = Pipeline([("preprocesado", preprocesador),
                      ("modelo", LogisticRegression(class_weight="balanced"))])

Objetivo: explicame, paso a paso y en orden de ejecucion real (no en el orden
en que estan escritas las lineas), que le pasa exactamente a una fila de datos
desde que entra en pipeline.fit(X_train, y_train) hasta que sale la prediccion
con pipeline.predict(X_test).

Restricciones: no des por hecho que conozco los nombres de los pasos - explica
que hace cada uno (SimpleImputer, RobustScaler, OneHotEncoder, ColumnTransformer,
Pipeline) con una frase cada uno antes de entrar en el orden de ejecucion.

Formato de salida: una lista numerada del recorrido de los datos, y al final
una frase explicando por que ajustar (fit) el preprocesado dentro del Pipeline,
en vez de hacerlo a mano antes del train_test_split, evita fugas de datos.
"""
print(prompt_explicar_pipeline)


**Qué se espera obtener**: una explicación ordenada por ejecución real (imputar → escalar en la rama numérica, codificar en la rama categórica, `ColumnTransformer` junta ambas ramas en una matriz, y esa matriz entra al modelo), y la idea de que `fit()` solo ocurre una vez, con el train, y `transform()` se reutiliza igual en el test.

**Qué debes verificar antes de darlo por bueno**: ejecuta cada paso del pipeline por separado (`preprocesador.fit_transform(X_train)`, mira la forma del resultado con `.shape`, compara el número de columnas antes y después del *one-hot*) y comprueba que coincide con lo que la IA ha descrito. Si la explicación menciona un paso que no está en el código (alucinación) o se salta uno, lo detectas exactamente así — ejecutando, no releyendo la explicación con más atención.

In [ ]:
# Verificacion real de un par de afirmaciones tipicas de la explicacion:
transformado = preprocesador.fit_transform(X_train)
print("Filas de X_train:", X_train.shape[0], " -> filas transformadas:", transformado.shape[0])
print("Columnas originales:", X_train.shape[1], " -> columnas tras ColumnTransformer:", transformado.shape[1])
print("(Sube el numero de columnas por el one-hot de las categoricas: cada categoria se convierte en una columna 0/1)")

### Prompt sugerido 2 — revisar un fragmento con leakage a propósito

In [ ]:
prompt_revisar_leakage = """
Contexto: un companero de curso ha escrito este codigo para preparar los
datos de un modelo de anulacion de polizas antes de entrenarlo:

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)  # X es el dataframe completo, antes de dividir

X_train, X_test, y_train, y_test = train_test_split(X_escalado, y, test_size=0.25)

modelo.fit(X_train, y_train)
print("Accuracy en test:", modelo.score(X_test, y_test))

Objetivo: revisa este codigo en busca de cualquier problema de fuga de datos
(data leakage) antes de que lo demos por bueno para el proyecto.

Restricciones: si encuentras un problema, explica exactamente por que lo es
(que informacion se esta filtrando y de donde a donde) y como se corregiria,
sin reescribir todo el proyecto, solo el fragmento necesario.

Formato de salida: primero un veredicto claro (tiene fuga de datos / no la
tiene), luego la explicacion, luego el fragmento corregido.
"""
print(prompt_revisar_leakage)


**Qué se espera obtener**: que la IA identifique que `scaler.fit_transform(X)` se ejecuta sobre `X` completo, antes del `train_test_split`, así que la media y desviación típica usadas para escalar "conocen" ya los datos que luego serán test — y que proponga mover el `fit` del scaler a después del split (ajustándolo solo con `X_train`), idealmente dentro de un `Pipeline`.

**Qué debes verificar antes de darlo por bueno**: comprueba tú mismo, como hicimos en la sección 6.2, que el fragmento corregido separa `fit` (solo train) de `transform` (train y test). Y — esta es la parte importante del ejercicio — si la IA con la que lo pruebes **no** detecta el problema, o lo explica de forma vaga ("siempre hay que dividir antes de todo" sin decir por qué), esa es exactamente la lección: un LLM no sustituye entender el concepto, porque puede fallar justo en detectar el error para el que se lo estás usando.

## Resumen

- `ColumnTransformer` aplica transformaciones distintas a columnas numéricas y categóricas, y `Pipeline` encadena preprocesado + modelo como un único objeto con `.fit()`/`.predict()`.
- Con clases desbalanceadas (11-12% de pólizas anuladas aquí), `class_weight="balanced"` es la alternativa nativa de scikit-learn a librerías externas como `imblearn`/SMOTE, fuera del alcance fijado para el curso.
- El `accuracy` solo no basta con clases desbalanceadas: precision y recall (y su lectura en términos de falso positivo/negativo de negocio) son las métricas que de verdad informan una decisión.
- `RandomizedSearchCV` busca hiperparámetros con validación cruzada sobre el `Pipeline` completo, para que el preprocesado también se reajuste correctamente en cada partición.
- El sobreajuste se detecta comparando métricas en train y test; una brecha grande es señal de simplificar el modelo.
- La fuga de datos (*data leakage*) no lanza ningún error — un `fit()` de un transformador ejecutado sobre todo el dataset antes de dividir en train/test puede inflar la métrica sin que se note a simple vista. La garantía es estructural: todo `fit()` va dentro del `Pipeline`, ajustado solo con el train.
- Una variable puede estar "bien calculada" y aun así ser una fuga si en producción no existiría en el momento de predecir — eso se detecta con criterio de negocio, no con código.
- Un modelo entrenado con datos históricos puede reproducir sesgos existentes; revisar el rendimiento por subgrupos y cuestionar las *proxy variables* es responsabilidad de quien construye el modelo, no algo que resuelva automáticamente scikit-learn ni la IA.

**Siguiente:** `03_ejemplo_aplicado_prudential.ipynb` — un segundo caso aplicado de ML en seguros (clasificación de riesgo de suscripción de vida, dataset público de Kaggle).